# Minimum Eigen Optimizer


Finding the solution to a Quadratic Unconstrained Binary Optimization(QUBO) problem is equivalent in finding the ground state of a corresponding Ising Hamiltonian - applicable to optimization, quantum chemistry and physics.
The binary variables {0,1} are replaced by spin variables with values {−1,+1}, which allows to replace the resulting spin variables by Pauli Z matrices, and thus, an Ising Hamiltonian. 
Details on this mapping -> [1].

Qiskit provides automatic conversion from a suitable QuadraticProgram to an Ising Hamiltonian, which then allows to leverage all the MinimumEigenSolver such as - VQE, - QAOA, or - NumpyMinimumEigensolver(classical exact method).

Qiskit wraps the translation to an Ising Hamiltonian (in Qiskit Aqua also called Operator), the call to an MinimumEigensolver as well as the translation of the results back to OptimizationResult in the MinimumEigenOptimizer.

In the following we first illustrate the conversion from a QuadraticProgram to an Operator and then show how to use the MinimumEigenOptimizer with different MinimumEigensolver to solve a given QuadraticProgram. The algorithms in Qiskit automatically try to convert a given problem to the supported problem class if possible, for instance, the MinimumEigenOptimizer will automatically translate integer variables to binary variables or add a linear equality constraints as a quadratic penalty term to the objective. It should be mentioned that Aqua will through a QiskitOptimizationError if conversion of a quadratic program with integer variable is attempted.

The circuit depth of QAOA potentially has to be increased with the problem size, which might be prohibitive for near-term quantum devices. A possible workaround is Recursive QAOA, as introduced in [2]. Qiskit generalizes this concept to the RecursiveMinimumEigenOptimizer, which is introduced at the end of this tutorial.
References

[1] [A. Lucas, Ising formulations of many NP problems, Front. Phys., 12 (2014).](https://arxiv.org/abs/1302.5843)

[2] [S. Bravyi, A. Kliesch, R. Koenig, E. Tang, Obstacles to State Preparation and Variational Optimization from Symmetry Protection, arXiv preprint arXiv:1910.08980 (2019).](https://arxiv.org/abs/1910.08980)


## Converting a QUBO to an Operator

In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')

In [ ]:
from qiskit_backend_config import BasicAer
from qiskit_optimization.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver
from qiskit_optimization.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler
from qiskit_optimization.algorithms import MinimumEigenOptimizer, RecursiveMinimumEigenOptimizer
from qiskit_optimization import QuadraticProgram

In [ ]:
# create a QUBO
# Note: this example used to have 32 variables (x01..x16, x21..x36). Both the exact
# NumPyMinimumEigensolver and a statevector QAOA need 2^n amplitudes - 2^32 is ~64 GB - so that
# version crashes the kernel. The first 16 variables (and all three quadratic terms) are kept.
qubo = QuadraticProgram()
for i in range(1, 17):
    qubo.binary_var(f'x{i:02d}')
qubo.minimize(linear=[1,-2,3,4,5,-6,7,8,9,10,-11,12,13,14,15,16],
              quadratic={('x01', 'x03'): 1, ('x11', 'x12'): -1, ('x14', 'x15'): 2})
print(qubo.export_as_lp_string())

In [ ]:
#Next we translate this QUBO into an Ising operator. 
#This results not only in an Operator but also in a constant offset to be taking into account 
#to shift the resulting value.

op, offset = qubo.to_ising()
print('offset: {}'.format(offset))
print('operator:')
print(op)

In [ ]:
#Sometimes a QuadraticProgram might also directly be given in the form of an Operator. 
#For such cases, Qiskit also provides a converter from an Operator back to a QuadraticProgram, 
#which we illustrate in the following.

from qiskit_optimization.translators import from_ising
qp = from_ising(op, offset, linear=True)
print(qp.export_as_lp_string())

In [ ]:
#This converter allows to translate an Operator to a QuadraticProgram and 
#then solve the problem with other algorithms that are not based on the Ising Hamiltonian representation, 
#such as the GroverOptimizer.

## Solving a QUBO with the MinimumEigenOptimizer

In [ ]:
#We start by initializing the MinimumEigensolver we want to use.
qaoa_mes = QAOA(sampler=StatevectorSampler(), optimizer=COBYLA())
exact_mes = NumPyMinimumEigensolver()

#Then, we use the MinimumEigensolver to create MinimumEigenOptimizer.
qaoa = MinimumEigenOptimizer(qaoa_mes)   # using QAOA
exact = MinimumEigenOptimizer(exact_mes)  # using the exact classical numpy minimum eigen solver

In [ ]:
#We first use the MinimumEigenOptimizer based on the classical exact 
#NumPyMinimumEigensolver to get the optimal benchmark solution for this small example.
exact_result = exact.solve(qubo)
print(exact_result)

In [ ]:
#Next we apply the MinimumEigenOptimizer based on QAOA to the same problem.
qaoa_result = qaoa.solve(qubo)
print(qaoa_result)

## RecursiveMinimumEigenOptimizer

In [ ]:
#The RecursiveMinimumEigenOptimizer takes a MinimumEigenOptimizer as input and 
#applies the recursive optimization scheme to reduce the size of the problem one variable at a time. 
#Once the size of the generated intermediate problem is below a given threshold (min_num_vars), 
#the RecursiveMinimumEigenOptimizer uses another solver (min_num_vars_optimizer), 
#e.g., an exact classical solver such as CPLEX or the MinimumEigenOptimizer based on the NumPyMinimumEigensolver.

#In the following, we show how to use the RecursiveMinimumEigenOptimizer 
#using the two MinimumEigenOptimizer introduced before.

#First, we construct the RecursiveMinimumEigenOptimizer 
#such that it reduces the problem size from 3 variables to 1 variable and 
#then uses the exact solver for the last variable. 
#Then we call solve to optimize the considered problem.

rqaoa = RecursiveMinimumEigenOptimizer(optimizer=qaoa, min_num_vars=5, min_num_vars_optimizer=exact)

rqaoa_result = rqaoa.solve(qubo)
print(rqaoa_result)